In [1]:
import os
import glob
import pandas as pd
import numpy as np

# ==========================================
# 1. 設定路徑與檔案清單
# ==========================================
# 將此路徑替換成你實際存放那些 172x172 CSV 檔案的資料夾
input_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses01/' 
output_filepath = os.path.join('/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS', 'SES01_AVG.csv')

# 使用 glob 自動抓取該資料夾下所有的 CSV 檔案
# 假設你的檔名長得像 sub-001_matrix.csv, sub-002_matrix.csv ...
file_list = glob.glob(os.path.join(input_dir, '*.csv'))

# 確保不要把可能已經算好的平均檔案又吃進去
file_list = [f for f in file_list if 'Average' not in f]

if not file_list:
    raise ValueError("找不到任何 CSV 檔案，請檢查路徑！")

print(f"找到 {len(file_list)} 個矩陣檔案，準備開始計算平均...")

# ==========================================
# 2. 讀取並累加所有矩陣
# ==========================================
# 先讀取第一個檔案，用來抓取「欄位名稱(Columns)」和「列名稱(Index)」，並作為累加的基底
first_df = pd.read_csv(file_list[0], index_col=0) # index_col=0 代表把第一欄當作 Row 標籤

# 取得維度資訊，預期為 (171, 171) -> 因為 index_col 扣掉了一欄標籤
expected_shape = first_df.shape 
region_names = first_df.columns

# 初始化一個全零的 NumPy array 來存放累加結果 (使用 NumPy 算比較快)
sum_matrix = np.zeros(expected_shape)

for idx, file in enumerate(file_list):
    # 讀取檔案
    df = pd.read_csv(file, index_col=0)
    
    # 安全性檢查：確保每個檔案的大小都一樣
    if df.shape != expected_shape:
        print(f"⚠️ 警告: 檔案 {file} 的大小為 {df.shape}，與第一個檔案 {expected_shape} 不同！已跳過。")
        continue
    
    # 將 Pandas 的數值提取出來並累加
    sum_matrix += df.values
    
    # 稍微印出進度
    if (idx + 1) % 10 == 0:
        print(f"已處理 {idx + 1} / {len(file_list)} 個檔案...")

# ==========================================
# 3. 計算平均
# ==========================================
# 將累加總和除以有效檔案的數量
avg_matrix = sum_matrix / len(file_list)

# ==========================================
# 4. 存回帶有標籤的 CSV 檔案
# ==========================================
# 將平均後的 NumPy array 包裝回 Pandas DataFrame，並貼上原本的欄列名稱
avg_df = pd.DataFrame(avg_matrix, index=first_df.index, columns=region_names)

# 儲存檔案
avg_df.to_csv(output_filepath, encoding='utf-8')

print(f"✅ 平均矩陣已成功儲存至: {output_filepath}")
print("前 5x5 的平均結果預覽：")
print(avg_df.iloc[:5, :5])

找到 44 個矩陣檔案，準備開始計算平均...
已處理 10 / 44 個檔案...
已處理 20 / 44 個檔案...
已處理 30 / 44 個檔案...
已處理 40 / 44 個檔案...
✅ 平均矩陣已成功儲存至: /bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/SES01_AVG.csv
前 5x5 的平均結果預覽：
                 Background  Precentral_L  Precentral_R  Frontal_Sup_2_L  \
Background              0.0      0.000000      0.000000         0.000000   
Precentral_L            0.0      0.000000      0.309617         0.197806   
Precentral_R            0.0      0.309617      0.000000         0.070805   
Frontal_Sup_2_L         0.0      0.197806      0.070805         0.000000   
Frontal_Sup_2_R         0.0      0.093643      0.133246         0.378095   

                 Frontal_Sup_2_R  
Background              0.000000  
Precentral_L            0.093643  
Precentral_R            0.133246  
Frontal_Sup_2_L         0.378095  
Frontal_Sup_2_R         0.000000  


In [2]:
import os
import glob
import pandas as pd
import numpy as np

# ==========================================
# 1. 設定路徑與檔案清單
# ==========================================
# 請將此路徑替換成你實際存放 CSV 檔案的資料夾
input_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses01/' 
output_filepath = os.path.join('/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS', 'SES01_AVG.csv')

file_list = glob.glob(os.path.join(input_dir, '*.csv'))
# 確保不要把可能已經算好的平均檔案又吃進去
file_list = [f for f in file_list if 'Average' not in f]

if not file_list:
    raise ValueError("找不到任何 CSV 檔案，請檢查路徑！")

print(f"找到 {len(file_list)} 個矩陣檔案，準備開始 Fisher's Z 轉換與平均...")

# ==========================================
# 2. 讀取並累加 Z 分數矩陣
# ==========================================
first_df = pd.read_csv(file_list[0], index_col=0)
expected_shape = first_df.shape 
region_names = first_df.columns

# 初始化一個全零矩陣來存放「Z 分數」的累加結果
sum_z_matrix = np.zeros(expected_shape)

for idx, file in enumerate(file_list):
    df = pd.read_csv(file, index_col=0)
    
    if df.shape != expected_shape:
        print(f"⚠️ 警告: 檔案 {file} 的大小為 {df.shape}，已跳過。")
        continue
    
    # 提取原始 r 值矩陣
    r_matrix = df.values
    
    # 【關鍵防呆】將 r 值稍微限制在 -0.999999 到 +0.999999 之間
    # 因為對角線的 r=1，如果直接算 arctanh(1) 會得到無限大 (inf)，導致後續平均壞掉
    r_matrix_clipped = np.clip(r_matrix, -0.999999, 0.999999)
    
    # 執行 Fisher's Z 轉換 (r -> Z)
    z_matrix = np.arctanh(r_matrix_clipped)
    
    # 將 Z 分數累加
    sum_z_matrix += z_matrix
    
    if (idx + 1) % 10 == 0:
        print(f"已處理 {idx + 1} / {len(file_list)} 個檔案...")

# ==========================================
# 3. 計算平均，並反轉回 r 值
# ==========================================
# 1. 計算 Z 分數的平均值
avg_z_matrix = sum_z_matrix / len(file_list)

# 2. 將平均 Z 分數反轉回 r 值 (Z -> r)
avg_r_matrix = np.tanh(avg_z_matrix)

# 3. (選擇性但強烈建議) 將對角線強制歸零
# 為了後續畫圖或算 Graph Theory (圖論) 指標方便，把自我相關的對角線設為 0
np.fill_diagonal(avg_r_matrix, 0)

# ==========================================
# 4. 存回帶有標籤的 CSV 檔案
# ==========================================
avg_df = pd.DataFrame(avg_r_matrix, index=first_df.index, columns=region_names)

avg_df.to_csv(output_filepath, encoding='utf-8')

print(f"✅ Fisher's Z 平均矩陣已成功儲存至: {output_filepath}")
print("前 5x5 的平均結果預覽：")
print(avg_df.iloc[:5, :5])

找到 44 個矩陣檔案，準備開始 Fisher's Z 轉換與平均...
已處理 10 / 44 個檔案...
已處理 20 / 44 個檔案...
已處理 30 / 44 個檔案...
已處理 40 / 44 個檔案...
✅ Fisher's Z 平均矩陣已成功儲存至: /bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/SES01_AVG.csv
前 5x5 的平均結果預覽：
                 Background  Precentral_L  Precentral_R  Frontal_Sup_2_L  \
Background              0.0      0.000000      0.000000         0.000000   
Precentral_L            0.0      0.000000      0.316716         0.204067   
Precentral_R            0.0      0.316716      0.000000         0.073344   
Frontal_Sup_2_L         0.0      0.204067      0.073344         0.000000   
Frontal_Sup_2_R         0.0      0.097627      0.138084         0.391777   

                 Frontal_Sup_2_R  
Background              0.000000  
Precentral_L            0.097627  
Precentral_R            0.138084  
Frontal_Sup_2_L         0.391777  
Frontal_Sup_2_R         0.000000  


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

# ==========================================
# 1. 設定雙資料夾路徑與嚴格配對
# ==========================================
pre_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses01'
post_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses02'

# 抓取兩邊的 CSV 檔案
pre_all = glob.glob(os.path.join(pre_dir, '*.csv'))
post_all = glob.glob(os.path.join(post_dir, '*.csv'))

# 擷取單純的檔名 (例如: 'sub-01.csv')
pre_names = set([os.path.basename(f) for f in pre_all])
post_names = set([os.path.basename(f) for f in post_all])

# 找出兩邊都有的共同檔名 (交集)，並進行排序確保順序一致
common_names = sorted(list(pre_names & post_names))
num_subjects = len(common_names)

print(f"✅ 在 ses01 找到 {len(pre_all)} 個檔案，在 ses02 找到 {len(post_all)} 個檔案。")
print(f"🔒 嚴格配對後，共有 {num_subjects} 位受試者同時擁有 Pre 與 Post 資料。")

if num_subjects != 44:
    print(f"⚠️ 注意：預期為 44 對，但實際配對成功數量為 {num_subjects} 對，請檢查是否有遺漏檔案！")

# 重新組合出精準對應的絕對路徑列表
pre_files = [os.path.join(pre_dir, name) for name in common_names]
post_files = [os.path.join(post_dir, name) for name in common_names]

# 讀取第一個檔案以獲取腦區名稱與維度
sample_df = pd.read_csv(pre_files[0], index_col=0)
region_names = sample_df.columns
n_rois = len(region_names)

# ==========================================
# 2. 萃取「右上半角」數值 (Z分數)
# ==========================================
row_idx, col_idx = np.triu_indices(n_rois, k=1)
num_edges = len(row_idx)

# 初始化儲存空間 shape: (受試者人數, 14706)
pre_data = np.zeros((num_subjects, num_edges))
post_data = np.zeros((num_subjects, num_edges))

for i in range(num_subjects):
    # 讀取已經是 Z 分數的數值矩陣
    pre_z = pd.read_csv(pre_files[i], index_col=0).values
    post_z = pd.read_csv(post_files[i], index_col=0).values
    
    # 抽出右上半角的連線
    pre_data[i, :] = pre_z[row_idx, col_idx]
    post_data[i, :] = post_z[row_idx, col_idx]

print(f"資料讀取完成。共有 {num_edges} 條獨立連線將進行 T 檢定。")

# ==========================================
# 3. 執行配對樣本 T 檢定 (Post 減 Pre)
# ==========================================
# T 值為正代表 Post > Pre (介入後連結增強)，為負代表 Post < Pre (介入後連結減弱)
t_stats, p_values = stats.ttest_rel(post_data, pre_data, axis=0)

# ==========================================
# 4. 執行 FDR 多重比較校正
# ==========================================
# alpha 設定為 0.05
reject, pval_corrected = fdrcorrection(p_values, alpha=0.05, method='indep')

# ==========================================
# 5. 整理結果並存檔
# ==========================================
results_list = []
for idx in range(num_edges):
    results_list.append({
        'Region_1': region_names[row_idx[idx]],
        'Region_2': region_names[col_idx[idx]],
        'T_value': t_stats[idx],
        'P_uncorrected': p_values[idx],
        'P_FDR': pval_corrected[idx],
        'Significant': reject[idx]
    })

df_results = pd.DataFrame(results_list)

# 篩選出通過 FDR 校正的顯著連線，並依照 p 值由小到大排序
df_significant = df_results[df_results['Significant'] == True].sort_values(by='P_FDR')

# 將結果儲存在 pre_dir 的上一層目錄中，方便管理
output_dir = os.path.dirname(pre_dir) 
df_results.to_csv(os.path.join(output_dir, 'All_Edges_T_test_Results.csv'), index=False)
df_significant.to_csv(os.path.join(output_dir, 'Significant_Edges_FDR05.csv'), index=False)

print("\n" + "="*50)
print(f"🎉 分析完成！在 {num_edges} 條連線中，")
print(f"發現了 {len(df_significant)} 條在介入前後有顯著差異 (FDR p < 0.05) 的連線。")
print(f"檔案已儲存至: {output_dir}")
print("="*50)

if len(df_significant) > 0:
    print("\n最強烈變化的前 10 條連線：")
    print(df_significant.head(10)[['Region_1', 'Region_2', 'T_value', 'P_FDR']])
else:
    print("\n⚠️ FDR 校正過於嚴格，沒有發現顯著連線。您可以觀察 P_uncorrected < 0.001 的結果，或嘗試 NBS 分析法。")

✅ 在 ses01 找到 44 個檔案，在 ses02 找到 44 個檔案。
🔒 嚴格配對後，共有 44 位受試者同時擁有 Pre 與 Post 資料。
資料讀取完成。共有 14535 條獨立連線將進行 T 檢定。

🎉 分析完成！在 14535 條連線中，
發現了 0 條在介入前後有顯著差異 (FDR p < 0.05) 的連線。
檔案已儲存至: /bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS

⚠️ FDR 校正過於嚴格，沒有發現顯著連線。您可以觀察 P_uncorrected < 0.001 的結果，或嘗試 NBS 分析法。


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

# ==========================================
# 1. 設定雙資料夾路徑與嚴格配對
# ==========================================
pre_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses01'
post_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses02'

# 抓取兩邊的 CSV 檔案
pre_all = glob.glob(os.path.join(pre_dir, '*.csv'))
post_all = glob.glob(os.path.join(post_dir, '*.csv'))

# 擷取單純的檔名 (例如: 'sub-01.csv')
pre_names = set([os.path.basename(f) for f in pre_all])
post_names = set([os.path.basename(f) for f in post_all])

# 找出兩邊都有的共同檔名 (交集)，並進行排序確保順序一致
common_names = sorted(list(pre_names & post_names))
num_subjects = len(common_names)

# print(f"✅ 在 ses01 找到 {len(pre_all)} 個檔案，在 ses02 找到 {len(post_all)} 個檔案。")
# print(f"🔒 嚴格配對後，共有 {num_subjects} 位受試者同時擁有 Pre 與 Post 資料。")

# if num_subjects != 44:
#     print(f"⚠️ 注意：預期為 44 對，但實際配對成功數量為 {num_subjects} 對，請檢查是否有遺漏檔案！")

# 重新組合出精準對應的絕對路徑列表
pre_files = [os.path.join(pre_dir, name) for name in common_names]
post_files = [os.path.join(post_dir, name) for name in common_names]

# 讀取第一個檔案以獲取腦區名稱與維度
sample_df = pd.read_csv(pre_files[0], index_col=0)
region_names = sample_df.columns
n_rois = len(region_names)

# ==========================================
# 2. 萃取「右上半角」數值並轉換為 Z 分數
# ==========================================
row_idx, col_idx = np.triu_indices(n_rois, k=1)
num_edges = len(row_idx)

# 初始化儲存空間 shape: (受試者人數, 14706)
pre_data = np.zeros((num_subjects, num_edges))
post_data = np.zeros((num_subjects, num_edges))

for i in range(num_subjects):
    # 1. 讀取原始的 Pearson r 數值矩陣
    pre_r = pd.read_csv(pre_files[i], index_col=0).values
    post_r = pd.read_csv(post_files[i], index_col=0).values
    
    # 2. 【關鍵安全機制】將 r 值限制在 -0.999999 到 0.999999 之間，避免 arctanh 產生 inf
    pre_r_clipped = np.clip(pre_r, -0.999999, 0.999999)
    post_r_clipped = np.clip(post_r, -0.999999, 0.999999)
    
    # 3. 執行 Fisher's Z 轉換 (r -> Z 分數)
    pre_z = np.arctanh(pre_r_clipped)
    post_z = np.arctanh(post_r_clipped)
    
    # 4. 抽出右上半角的連線存入統計陣列
    pre_data[i, :] = pre_z[row_idx, col_idx]
    post_data[i, :] = post_z[row_idx, col_idx]

# print(f"資料讀取與 Fisher's Z 轉換完成。共有 {num_edges} 條獨立連線將進行 T 檢定。")

# ==========================================
# 3. 執行配對樣本 T 檢定 (Post 減 Pre)
# ==========================================
# T 值為正代表 Post > Pre (介入後連結增強)，為負代表 Post < Pre (介入後連結減弱)
t_stats, p_values = stats.ttest_rel(post_data, pre_data, axis=0)

# ==========================================
# 4. 執行 FDR 多重比較校正
# ==========================================
# alpha 設定為 0.05
reject, pval_corrected = fdrcorrection(p_values, alpha=0.05, method='indep')

# ==========================================
# 5. 整理結果並存檔
# ==========================================
results_list = []
for idx in range(num_edges):
    results_list.append({
        'Region_1': region_names[row_idx[idx]],
        'Region_2': region_names[col_idx[idx]],
        'T_value': t_stats[idx],
        'P_uncorrected': p_values[idx],
        'P_FDR': pval_corrected[idx],
        'Significant_FDR': reject[idx] # 稍微改名避免混淆
    })

df_results = pd.DataFrame(results_list)

# 🌟 關鍵修改：改為篩選 P_uncorrected < 0.001 的連線，並依照原始 p 值排序
df_significant_uncorr = df_results[df_results['P_uncorrected'] < 0.001].sort_values(by='P_uncorrected')

# 將結果儲存在 pre_dir 的上一層目錄中，方便管理
output_dir = os.path.dirname(pre_dir) 

# 存檔名稱也建議修改，以免之後看檔名搞混
df_results.to_csv(os.path.join(output_dir, 'All_Edges_T_test_Results.csv'), index=False)
df_significant_uncorr.to_csv(os.path.join(output_dir, 'Significant_Edges_Uncorr001.csv'), index=False)

print("\n" + "="*50)
# print(f"🎉 分析完成！在 {num_edges} 條連線中，")
# print(f" {len(df_significant_uncorr)}  (Uncorrected p < 0.001) ")
# print(f"檔案已儲存至: {output_dir}")
# print("="*50)

if len(df_significant_uncorr) > 0:
    # 列印時改為顯示 P_uncorrected
    print(df_significant_uncorr.head(10)[['Region_1', 'Region_2', 'T_value', 'P_uncorrected']])




                 Region_1       Region_2   T_value  P_uncorrected
323          Precentral_L      ACC_sup_L  4.304082       0.000095
324          Precentral_L      ACC_sup_R  3.787645       0.000468
4437             OFCant_R       Heschl_R  3.767919       0.000496
12850     Cerebellum_10_R     Thal_LGN_L -3.758622       0.000510
13980          Thal_MDl_L     Thal_PuI_R  3.711775       0.000587
12090      Cerebellum_3_R     Thal_MGN_L -3.702755       0.000603
4069             OFCmed_L     Thal_PuM_L  3.613527       0.000786
4340             OFCant_L     Thal_VPL_L  3.567618       0.000900
4483             OFCant_R     Thal_VPL_R  3.556810       0.000929
1222   Frontal_Inf_Oper_L  Postcentral_L  3.539588       0.000977


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection


pre_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses01'
post_dir = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/FOR_BHS/CSVS/ses02'


pre_all = glob.glob(os.path.join(pre_dir, '*.csv'))
post_all = glob.glob(os.path.join(post_dir, '*.csv'))

pre_names = set([os.path.basename(f) for f in pre_all])
post_names = set([os.path.basename(f) for f in post_all])

common_names = sorted(list(pre_names & post_names))
num_subjects = len(common_names)

print(f"✅ 在 ses01 找到 {len(pre_all)} 個檔案，在 ses02 找到 {len(post_all)} 個檔案。")
print(f"🔒 嚴格配對後，共有 {num_subjects} 位受試者同時擁有 Pre 與 Post 資料。")

pre_files = [os.path.join(pre_dir, name) for name in common_names]
post_files = [os.path.join(post_dir, name) for name in common_names]

sample_df = pd.read_csv(pre_files[0], index_col=0)
region_names = sample_df.columns
n_rois = len(region_names)

# ==========================================
# 2. 萃取「右上半角」數值 (Z分數)
# ==========================================
row_idx, col_idx = np.triu_indices(n_rois, k=1)
num_edges = len(row_idx)

# 初始化儲存空間 shape: (受試者人數, 14706)
pre_data = np.zeros((num_subjects, num_edges))
post_data = np.zeros((num_subjects, num_edges))

for i in range(num_subjects):
    # 讀取已經是 Z 分數的數值矩陣
    pre_z = pd.read_csv(pre_files[i], index_col=0).values
    post_z = pd.read_csv(post_files[i], index_col=0).values
    
    # 抽出右上半角的連線
    pre_data[i, :] = pre_z[row_idx, col_idx]
    post_data[i, :] = post_z[row_idx, col_idx]

print(f"資料讀取完成。共有 {num_edges} 條獨立連線將進行 T 檢定。")

# ==========================================
# 3. 執行配對樣本 T 檢定 (Post 減 Pre)
# ==========================================
# T 值為正代表 Post > Pre (介入後連結增強)，為負代表 Post < Pre (介入後連結減弱)
t_stats, p_values = stats.ttest_rel(post_data, pre_data, axis=0)

# ==========================================
# 4. 執行 FDR 多重比較校正
# ==========================================
# alpha 設定為 0.05
reject, pval_corrected = fdrcorrection(p_values, alpha=0.05, method='indep')

# ==========================================
# 5. 整理結果並存檔
# ==========================================
results_list = []
for idx in range(num_edges):
    results_list.append({
        'Region_1': region_names[row_idx[idx]],
        'Region_2': region_names[col_idx[idx]],
        'T_value': t_stats[idx],
        'P_uncorrected': p_values[idx],
        'P_FDR': pval_corrected[idx],
        'Significant_FDR': reject[idx] # 稍微改名避免混淆
    })

df_results = pd.DataFrame(results_list)

# 🌟 關鍵修改：改為篩選 P_uncorrected < 0.001 的連線，並依照原始 p 值排序
df_significant_uncorr = df_results[df_results['P_uncorrected'] < 0.001].sort_values(by='P_uncorrected')

# 將結果儲存在 pre_dir 的上一層目錄中，方便管理
output_dir = os.path.dirname(pre_dir) 

# 存檔名稱也建議修改，以免之後看檔名搞混
df_results.to_csv(os.path.join(output_dir, 'All_Edges_T_test_Results.csv'), index=False)
df_significant_uncorr.to_csv(os.path.join(output_dir, 'Significant_Edges_Uncorr001.csv'), index=False)

print("\n" + "="*50)
print(f"🎉 分析完成！在 {num_edges} 條連線中，")
print(f"發現了 {len(df_significant_uncorr)} 條在介入前後有顯著差異 (Uncorrected p < 0.001) 的連線。")
print(f"檔案已儲存至: {output_dir}")
print("="*50)

if len(df_significant_uncorr) > 0:
    print("\n最強烈變化的前 10 條連線：")
    # 列印時改為顯示 P_uncorrected
    print(df_significant_uncorr.head(10)[['Region_1', 'Region_2', 'T_value', 'P_uncorrected']])
else:
    print("\n⚠️ 即使放寬到 Uncorrected p < 0.001，依然沒有發現顯著連線。")